## FLOSSK Historic Full Sync

This notebook performs a full sync against the historic FLOSSK sources.

It has two parts:
- `gazetat/`: searchable newspaper archive, exported as sarcasm bootstrap CSV
- `librat/`: books catalog sync, exported as a metadata CSV

Important: the labels are heuristic bootstrap labels, not final gold annotations.

### Endpoint Log

The newspaper search endpoint is a same-page HTML form POST, not a JSON API:

- URL: `POST https://books.flossk.org/gazetat/`
- Key form fields: `pages`, `startDate`, `endDate`, `textQuery`, `repo`, `formId=search`, `context`, `insensitive`, `currentPDF`, `pdfPage`, `currentPage`
- Response type: HTML
- Result structure: jQuery accordion rows containing OCR snippet, PDF filename, date, and page number
- Viewer URL pattern: `https://books.flossk.org/wp-content/plugins/gazetat/js/pdfjs/web/viewer.html?file=../../../repo/<PDF>&search=<TERM>`

In [ ]:
from pathlib import Path
import importlib.util
import json

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "scripts").exists():
    for parent in BASE_DIR.parents:
        if (parent / "scripts").exists():
            BASE_DIR = parent
            break

SCRIPTS_DIR = BASE_DIR / "scripts"

UTILS_PATH = SCRIPTS_DIR / "03b_flossk_historical_sources_utils.py"
if not UTILS_PATH.exists():
    raise FileNotFoundError(f"Could not find helper script: {UTILS_PATH}")

spec = importlib.util.spec_from_file_location("historical_sources_utils", UTILS_PATH)
historical_sources_utils = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(historical_sources_utils)

GAZETA_URL = historical_sources_utils.GAZETA_URL
NEGATIVE_SEARCH_TERMS = historical_sources_utils.NEGATIVE_SEARCH_TERMS
POSITIVE_SEARCH_TERMS = historical_sources_utils.POSITIVE_SEARCH_TERMS
build_bootstrap_dataset = historical_sources_utils.build_bootstrap_dataset
export_books_catalog_csv = historical_sources_utils.export_books_catalog_csv
extract_books_catalog = historical_sources_utils.extract_books_catalog
fetch_html = historical_sources_utils.fetch_html
parse_repo_date_ranges = historical_sources_utils.parse_repo_date_ranges


In [2]:
index_html = fetch_html(GAZETA_URL)
repo_ranges = parse_repo_date_ranges(index_html)

endpoint_snapshot = {
    "url": GAZETA_URL,
    "method": "POST",
    "repos": repo_ranges,
    "positive_search_terms": POSITIVE_SEARCH_TERMS,
    "negative_search_terms": NEGATIVE_SEARCH_TERMS,
}

print(json.dumps(endpoint_snapshot, indent=2, ensure_ascii=False)[:4000])


{
  "url": "https://books.flossk.org/gazetat/",
  "method": "POST",
  "repos": {
    "bujku": [
      "1991-01-18",
      "1998-12-30"
    ],
    "rilindja": [
      "1948-01-08",
      "1988-12-30"
    ]
  },
  "positive_search_terms": [
    "ironi",
    "satir",
    "sarkaz",
    "parodi",
    "thumb",
    "tallje",
    "qesharak",
    "grotesk",
    "absurd"
  ],
  "negative_search_terms": [
    "kosove",
    "qeveri",
    "arsim",
    "ekonomi",
    "shkolle",
    "sport",
    "zgjedhje",
    "fshat",
    "bujqesi"
  ]
}


In [3]:
selected_repos = None  # None means full sync across all newspaper repos
max_per_label = 600

detailed_df, final_df = build_bootstrap_dataset(
    max_per_label=max_per_label,
    repos=selected_repos,
)

print(final_df["is_sarcasm"].value_counts())
final_df.head(10)


is_sarcasm
no     600
yes    600
Name: count, dtype: int64


,text,is_sarcasm
0,"Shtërpcës. Për prindërit, mysa- firët e arsimt...",no
1,Demonstratat e vitit 1981 ishin plebishiti i p...,no
2,gjoja milicia nuk paska mundur të më gjejë për...,yes
3,"dhemb, e syni gjum' nuk ka...” bota e fjalëve ...",yes
4,se Sutjeska (që ia zuri vendin Prishtinës) për...,yes
5,"në shkollën fillore të Vitomiricës, organet e ...",no
6,harmonioze. Është ironi e fatit që intelektual...,yes
7,I UA TRAVEL AIR don PN 21 aifu KYS Qep ue PRIS...,no
8,(Vijon prej faqës së parë) Bonit konstaton me ...,yes
9,"eksport Kosova zinte vendin e parë, përkatësis...",no


In [4]:
base_dir = Path.cwd().resolve()
if not (base_dir / "data").exists():
    for parent in base_dir.parents:
        if (parent / "data").exists():
            base_dir = parent
            break

output_csv = base_dir / "data" / "sarcasm_flossk_historic_bootstrap.csv"
final_df.to_csv(output_csv, index=False)
print(output_csv)
print(final_df.shape)


/Users/bleronaidrizi/Sources/Master_Tema_e_Diplomes/Punimi/Sarcasm-Detection-Albanian-News-Dataset/data/sarcasm_flossk_historic_bootstrap.csv
(1200, 2)


### Librat Full Sync

This section syncs the visible `librat/` catalog and exports all detected book entries into one CSV.

In [3]:
books_sample = extract_books_catalog(limit=5)
books_sample


[{'book_page': 'https://books.flossk.org/2020/10/26/pjeter-pan/',
  'title': 'Pjetër Pan',
  'description': 'Pjetër Pan - Xhejms Barri "Rilindja" Prishtinë 1959 Përkthyes: Ramiz Kelmendi Faqe 105',
  'description_lines': 'Pjetër Pan - Xhejms Barri "Rilindja" Prishtinë 1959 Përkthyes: Ramiz Kelmendi Faqe 105',
  'pdf_url': 'https://books.flossk.org/wp-content/uploads/2020/10/1959-xhejms-barri-pjeter-pani.pdf',
  'thumbnail_url': 'https://books.flossk.org/wp-content/uploads/2020/10/Screenshot-from-2020-10-26-131240.png',
  'published_at': '2020-10-26T15:23:46+00:00',
  'updated_at': '2022-11-15T12:01:41+00:00',
  'author': 'Valmir Mustafa'},
 {'book_page': 'https://books.flossk.org/2020/10/26/ajvanho/',
  'title': 'Ajvanho',
  'description': 'Ajvanho - Valter Skot "Rilindja" Prishtinë 1959 Përkthyes: Latif Mulaku dhe Latif Berisha Faqe 130',
  'description_lines': 'Ajvanho - Valter Skot "Rilindja" Prishtinë 1959 Përkthyes: Latif Mulaku dhe Latif Berisha Faqe 130',
  'pdf_url': 'https://b

In [4]:
books_df, books_csv = export_books_catalog_csv()
print(books_csv)
print(books_df.shape)
books_df.head(10)


/Users/bleronaidrizi/Sources/Master_Tema_e_Diplomes/Punimi/Sarcasm-Detection-Albanian-News-Dataset/data/flossk_books_catalog.csv
(50, 9)


,book_page,title,description,description_lines,pdf_url,thumbnail_url,published_at,updated_at,author
0,https://books.flossk.org/2020/10/26/pjeter-pan/,Pjetër Pan,"Pjetër Pan - Xhejms Barri ""Rilindja"" Prishtinë...","Pjetër Pan - Xhejms Barri ""Rilindja"" Prishtinë...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:23:46+00:00,2022-11-15T12:01:41+00:00,Valmir Mustafa
1,https://books.flossk.org/2020/10/26/ajvanho/,Ajvanho,"Ajvanho - Valter Skot ""Rilindja"" Prishtinë 195...","Ajvanho - Valter Skot ""Rilindja"" Prishtinë 195...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:22:10+00:00,2022-11-15T12:02:44+00:00,Valmir Mustafa
2,https://books.flossk.org/2020/10/26/mbi-krahet...,Mbi krahët e fluturës,"Mbi krahët e fluturës - Rexep Hoxha ""Rilindja""...","Mbi krahët e fluturës - Rexep Hoxha ""Rilindja""...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:19:31+00:00,2022-11-15T12:02:51+00:00,Valmir Mustafa
3,https://books.flossk.org/2020/10/26/hajdi/,Hajdi,"Hajdi - Johana Shpiri ""Rilindja"" Prishtinë 195...","Hajdi - Johana Shpiri ""Rilindja"" Prishtinë 195...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:17:38+00:00,2022-11-15T12:02:55+00:00,Valmir Mustafa
4,https://books.flossk.org/2020/10/26/era-dhe-ko...,Era dhe kolona,"Era dhe kolona - Hivzi Sylejmani ""Rilindja"" Pr...","Era dhe kolona - Hivzi Sylejmani ""Rilindja"" Pr...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:15:47+00:00,2022-11-15T12:03:00+00:00,Valmir Mustafa
5,https://books.flossk.org/2020/10/26/zoja-bovari/,Zoja Bovari,"Zoja Bovari - Gustav Flober ""Rilindja"" Prishti...","Zoja Bovari - Gustav Flober ""Rilindja"" Prishti...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:12:34+00:00,2022-11-15T12:03:06+00:00,Valmir Mustafa
6,https://books.flossk.org/2020/10/26/udhetimi-n...,Udhëtimi në Kongo,"Udhëtimi në Kongo - Andre Zhid ""Rilindja"" Pris...","Udhëtimi në Kongo - Andre Zhid ""Rilindja"" Pris...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T15:10:18+00:00,2022-11-15T12:03:11+00:00,Valmir Mustafa
7,https://books.flossk.org/2020/10/26/vocrraku/,Vocrraku,"Vocrraku - Alfons Dode ""Rilindja"" Prishtinë 19...","Vocrraku - Alfons Dode ""Rilindja"" Prishtinë 19...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T14:32:52+00:00,2022-11-15T12:03:16+00:00,Valmir Mustafa
8,https://books.flossk.org/2020/10/26/thembra-e-...,Thembra e hekurt,"Thembra e hekurt - Xhek London ""Rilndija"" Pris...","Thembra e hekurt - Xhek London ""Rilndija"" Pris...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T14:28:50+00:00,2022-11-15T12:03:22+00:00,Valmir Mustafa
9,https://books.flossk.org/2020/10/26/rruga-ime/,Rruga ime,"Rruga Ime (vjersha) - Sergej Jesenjin ""Rilindj...","Rruga Ime (vjersha) - Sergej Jesenjin ""Rilindj...",https://books.flossk.org/wp-content/uploads/20...,https://books.flossk.org/wp-content/uploads/20...,2020-10-26T14:22:01+00:00,2022-11-15T12:03:26+00:00,Valmir Mustafa
